In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)

print("Dataset loaded succesfull!y")
print("Shape:",df.shape)

df.head

Dataset loaded succesfull!y
Shape: (891, 12)


<bound method NDFrame.head of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                     

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
print("Exact duplicate rows:",df.duplicated().sum())

print("Duplicate passenger IDs:",df["PassengerId"].duplicated().sum())

Exact duplicate rows: 0
Duplicate passenger IDs: 0


In [5]:
print("Sex values:", df["Sex"].unique())
print("Embarked values:",df["Embarked"].unique())

Sex values: ['male' 'female']
Embarked values: ['S' 'C' 'Q' nan]


### Data Cleaning

In [6]:
# Keep the original data unchanged
cleaned_df = df.copy()

# 1. Remove exact duplicate records
cleaned_df = cleaned_df.drop_duplicates()

# 2. Remove unnecessary spaces from text columns
text_columns = cleaned_df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    cleaned_df[col] = cleaned_df[col].str.strip()

# 3. Standardize categorical values
cleaned_df["Sex"] = (
    cleaned_df["Sex"].str.lower()
)

cleaned_df["Embarked"] = (
    cleaned_df["Embarked"].str.upper()
)

# 4. Convert numerical columns to numeric types
numeric_columns = [
    "PassengerId", "Survived", "Pclass",
    "Age", "SibSp", "Parch", "Fare"
]

for col in numeric_columns:
    cleaned_df[col] = pd.to_numeric(
        cleaned_df[col], errors="coerce"
    )

# 5. Treat impossible negative values as missing
for col in ["Age", "Fare", "SibSp", "Parch"]:
    cleaned_df.loc[cleaned_df[col] < 0, col] = np.nan

# 6. Fill missing ages using the median for
# passengers in the same class and sex group
cleaned_df["Age"] = cleaned_df.groupby(
    ["Pclass", "Sex"]
)["Age"].transform(
    lambda group: group.fillna(group.median())
)

# If any age is still missing, use the overall median
cleaned_df["Age"] = cleaned_df["Age"].fillna(
    cleaned_df["Age"].median()
)

# 7. Fill missing Embarked values with the mode
cleaned_df["Embarked"] = cleaned_df["Embarked"].fillna(
    cleaned_df["Embarked"].mode()[0]
)

# 8. Fill missing fares with the median fare
cleaned_df["Fare"] = cleaned_df["Fare"].fillna(
    cleaned_df["Fare"].median()
)

# 9. Cabin has a large number of missing values.
# Remove it and document this decision in README.md
cleaned_df = cleaned_df.drop(columns=["Cabin"])

print("Cleaning completed.")
cleaned_df.head()

Cleaning completed.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0.0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0.0,0.0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0.0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0.0,0.0,373450,8.0500,S


In [10]:
print("Original shape:",df.shape)
print("Cleaned shape:",cleaned_df.shape)

print("\nMissing values after cleaning:")
print(cleaned_df.isnull().sum())

print("\nRemaining exact duplicates:",cleaned_df.duplicated().sum())

print("\nFinal data types:")
print(cleaned_df.dtypes)

print("\nFirst 5 cleaned rows:")
display(cleaned_df.head())

Original shape: (891, 12)
Cleaned shape: (891, 11)

Missing values after cleaning:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Remaining exact duplicates: 0

Final data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp          float64
Parch          float64
Ticket          object
Fare           float64
Embarked        object
dtype: object

First 5 cleaned rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0.0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0.0,0.0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0.0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0.0,0.0,373450,8.0500,S


In [8]:
from pathlib import Path

output_folder = Path("data")
output_folder.mkdir(exist_ok=True)

cleaned_df.to_csv(output_folder / "titanic_cleaned.csv",index=False)

print("Cleaned dataset saved successfully!")


Cleaned dataset saved successfully!


In [14]:
print(cleaned_df.dtypes)
saved_df = pd.read_csv("data/titanic_cleaned.csv")

print("\nSaved dataset shape:", saved_df.shape)
print("Missing values in saved file:", saved_df.isnull().sum().sum())
print("Duplicate rows in saved file:", saved_df.duplicated().sum())

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object

Saved dataset shape: (891, 11)
Missing values in saved file: 0
Duplicate rows in saved file: 0


In [15]:
# Convert passenger counts to integers
cleaned_df["SibSp"] = cleaned_df["SibSp"].astype("int64")
cleaned_df["Parch"] = cleaned_df["Parch"].astype("int64")

# Save the final cleaned dataset
cleaned_df.to_csv("data/titanic_cleaned.csv", index=False)

print(cleaned_df.dtypes)

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object


In [16]:
from pathlib import Path

output_folder = Path("data")
output_folder.mkdir(exist_ok=True)

cleaned_df.to_csv(
    output_folder / "titanic_cleaned.csv",
    index=False
)

print("Final cleaned dataset saved successfully!")
print("Shape:", cleaned_df.shape)
print("Missing values:", cleaned_df.isnull().sum().sum())
print("Duplicate rows:", cleaned_df.duplicated().sum())

Final cleaned dataset saved successfully!
Shape: (891, 11)
Missing values: 0
Duplicate rows: 0
